In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. VERİ OKUMA AŞAMASI (EXTRACT)

print("Ham veri CSV'den okunuyor...")
try:
    df_ham = pd.read_csv("ham_fiyatlar.csv") 
except FileNotFoundError:
    print("Hata: 'ham_fiyatlar.csv' dosyası bulunamadı. Lütfen dosyanın kodla aynı klasörde olduğundan emin ol.")
    exit()

# 2. VERİ İŞLEME VE DÖNÜŞTÜRME AŞAMASI (TRANSFORM)
df_islenmis = pd.DataFrame()
df_islenmis['A_1'] = df_ham['Fiyat'].shift(3)  # T-2 (2 dönem önce)
df_islenmis['A_2'] = df_ham['Fiyat'].shift(2)  # T-1 (1 dönem önce)
df_islenmis['A_3'] = df_ham['Fiyat'].shift(1)  # T   (Bugün)
df_islenmis['Tahmin'] = df_ham['Fiyat']        # T+1 (Yarın - Hedef)

# Shift işlemi nedeniyle oluşan NaN (boş) satırları temizle
df_islenmis = df_islenmis.dropna().reset_index(drop=True)


# 3. İŞLENMİŞ VERİYİ YENİ CSV OLARAK KAYDETME (LOAD)

df_islenmis.to_csv("islenmis_egitim_matrisi.csv", index=False)
print(">>> BAŞARILI: Veri kaydırıldı ve 'islenmis_egitim_matrisi.csv' olarak klasöre kaydedildi!\n")

# 4. MAKİNE ÖĞRENMESİ AŞAMASI (TRAIN & EVALUATE)
X = df_islenmis[['A_1', 'A_2', 'A_3']]
y = df_islenmis['Tahmin']

modeller = {
    "Dogrusal Regresyon (Linear)": LinearRegression(),
    "Ridge Regresyon (L2)": Ridge(alpha=1.0),
    "Lasso Regresyon (L1)": Lasso(alpha=0.1)
}

for model_adi, model in modeller.items():
    print(f"{'='*50}\nMODEL: {model_adi}\n{'='*50}")
    
    # Eğit ve Tahmin Et
    model.fit(X, y)
    y_tahmin = model.predict(X)
    
    # Hata ve Başarı Metrikleri
    r2 = r2_score(y, y_tahmin)
    mse = mean_squared_error(y, y_tahmin)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, y_tahmin)
    
    print(f"R^2 Skoru : {r2:.4f}")
    print(f"MSE       : {mse:.4f}")
    print(f"RMSE      : {rmse:.4f}")
    print(f"MAE       : {mae:.4f}")
    
    # Kesisim yerine model.intercept_ yazdırıyoruz
    print(f"\nmodel.intercept_ : {model.intercept_:.4f}")
    
    # Katsayılar yerine model.coef_ yazdırıyoruz
    print("model.coef_ :")
    for sutun, katsayi in zip(X.columns, model.coef_):
        print(f"  {sutun}: {katsayi:.4f}")
        
    # Standart Hata hesaplaması ve yazdırılması
    if model_adi == "Dogrusal Regresyon (Linear)":
        n = len(y)
        p = X.shape[1]
        X_matrisi = np.column_stack((np.ones(n), X))
        artiklar = y - y_tahmin
        sigma_kare = np.sum(artiklar**2) / (n - p - 1)
        varyans_kovaryans_matrisi = np.linalg.inv(np.dot(X_matrisi.T, X_matrisi)) * sigma_kare
        standart_hatalar = np.sqrt(np.diagonal(varyans_kovaryans_matrisi))
        
        # Standart hatalar yerine std_err yazdırıyoruz
        print("\nstd_err :")
        print(f"  Intercept: {standart_hatalar[0]:.4f}")
        for i, sutun in enumerate(X.columns):
            print(f"  {sutun}: {standart_hatalar[i+1]:.4f}")

Ham veri CSV'den okunuyor...
>>> BAŞARILI: Veri kaydırıldı ve 'islenmis_egitim_matrisi.csv' olarak klasöre kaydedildi!

MODEL: Dogrusal Regresyon (Linear)
R^2 Skoru : 0.2195
MSE       : 604.3572
RMSE      : 24.5837
MAE       : 16.8495

model.intercept_ : 49.0947
model.coef_ :
  A_1: -0.0458
  A_2: -0.0715
  A_3: 0.5065

std_err :
  Intercept: 30.8348
  A_1: 0.2617
  A_2: 0.3025
  A_3: 0.2712
MODEL: Ridge Regresyon (L2)
R^2 Skoru : 0.2195
MSE       : 604.3572
RMSE      : 24.5837
MAE       : 16.8495

model.intercept_ : 49.0969
model.coef_ :
  A_1: -0.0458
  A_2: -0.0715
  A_3: 0.5064
MODEL: Lasso Regresyon (L1)
R^2 Skoru : 0.2195
MSE       : 604.3572
RMSE      : 24.5837
MAE       : 16.8487

model.intercept_ : 49.0893
model.coef_ :
  A_1: -0.0458
  A_2: -0.0713
  A_3: 0.5062
